### Ingest drivers.json file
1. Read the file using Spark dataframe reader API
2. Add metadata columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType



name_schema = StructType([
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True)
])

drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", name_schema, True),
    StructField("dateOfBirth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
drivers_df = (
    spark.read
    .format("json")
    .schema(drivers_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
#display(drivers_df)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
write_to_bronze(
    drivers_final_df,
    table_name,
    v_batch_id
)

In [0]:
#display(spark.table(table_name))

In [0]:
%sql
--select * from formula1|_incr.bronze.drivers